# Gate 1 — Discovery

Finds which ICT and NBBTRADER videos exist, filters ICT down to the eight topics, and reports the counts.

**Nothing is installed on your computer.** This runs on Google's machines, in your browser.

---

## How to run it

1. Menu at the top → **Runtime** → **Run all**
2. If a box pops up saying the notebook wasn't authored by Google, click **Run anyway**
3. Wait. It takes roughly 5–15 minutes.
4. Scroll to the bottom, copy the **GATE 1** report, and paste it back into the chat.

You don't need to understand or edit any of the code below.

---

### One thing that might go wrong

YouTube sometimes blocks Google's datacentre addresses and asks the requester to "sign in to confirm you're not a bot". If that happens the notebook will say so plainly in **Step 4** and tell you the fix. Listing videos (what this notebook does) is usually allowed even when downloading is not, so it will most likely be fine.

## Step 1 — Install the two tools this needs

In [ ]:
%pip install -q --upgrade yt-dlp pydantic

import yt_dlp, pydantic
print("yt-dlp ", yt_dlp.version.__version__)
print("pydantic", pydantic.VERSION)
print("\nStep 1 done.")

## Step 2 — Download the pipeline code

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/dboy140/Dboytrades.git"
BRANCH = "claude/ict-nbbtrader-trading-system-43hipg"

# Start clean so re-running the notebook is always safe.
if os.path.isdir("/content/Dboytrades"):
    shutil.rmtree("/content/Dboytrades")

r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/Dboytrades"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise SystemExit(f"Could not download the code:\n{r.stderr}")

os.chdir("/content/Dboytrades")
print("Step 2 done. Code downloaded.")

## Step 3 — Quick self-test

Runs the test suite. This needs no internet and proves the code is intact before spending time on the real run.

In [ ]:
%pip install -q pytest
!python -m pytest -q 2>&1 | tail -5

## Step 4 — Can this machine reach YouTube?

In [ ]:
import subprocess

probe = subprocess.run(
    ["yt-dlp", "--flat-playlist", "--dump-json", "--playlist-end", "1",
     "https://www.youtube.com/channel/UCtjxa77NqamhVC8atV85Rog/videos"],
    capture_output=True, text=True, timeout=180,
)

err = probe.stderr.lower()
if probe.returncode == 0 and probe.stdout.strip():
    print("YouTube is reachable. Continue to Step 5.")
elif "sign in to confirm" in err or "bot" in err:
    print("BLOCKED: YouTube is challenging this Colab machine as a bot.\n")
    print("Fix: reconnect to a different machine and try again --")
    print("  Runtime -> Disconnect and delete runtime, then Runtime -> Run all.")
    print("  Colab hands out a different address each time, so this often clears it.")
    print("\nIf it keeps happening, tell me in the chat and I'll give you a")
    print("cookie-based workaround.")
else:
    print("Could not reach YouTube. Raw error below -- paste this into the chat:\n")
    print(probe.stderr[:1500])

## Step 5 — Run discovery

This is the real work: listing every ICT video, filtering to the eight topics, listing everything from NBBTRADER, checking both candidate NBBTRADER channels, and searching for his guest appearances.

Expect 5–15 minutes. Output appears as it goes.

In [ ]:
!python -m scripts.discover

## Step 6 — What got filtered out

A sample of ICT videos the eight-topic filter removed. Skim for anything you'd want kept.

In [ ]:
import json, pathlib

p = pathlib.Path("data/excluded.json")
if not p.exists():
    print("No excluded.json -- discovery did not finish. Check Step 5 output.")
else:
    rows = json.loads(p.read_text())
    print(f"{len(rows)} ICT videos excluded. First 40 titles:\n")
    for r in rows[:40]:
        print(" -", r.get("title", "")[:95])
    if len(rows) > 40:
        print(f"\n... and {len(rows) - 40} more (full list in the download below).")

## Step 7 — Download the results

Saves a zip to your computer. Attach it in the chat along with the Gate 1 report.

In [ ]:
import shutil, pathlib

out = pathlib.Path("/content/gate1_results")
if out.exists():
    shutil.rmtree(out)
out.mkdir()

copied = []
for src in ["data/manifest.json", "data/excluded.json",
            "data/channel_probe.json", "logs/gate1_report.json"]:
    p = pathlib.Path(src)
    if p.exists():
        shutil.copy(p, out / p.name)
        copied.append(p.name)

if not copied:
    print("Nothing to download -- discovery did not produce output.")
else:
    shutil.make_archive("/content/gate1_results", "zip", out)
    print("Packaged:", ", ".join(copied))
    try:
        from google.colab import files
        files.download("/content/gate1_results.zip")
    except Exception as exc:
        print(f"\nAuto-download unavailable ({exc}).")
        print("Use the folder icon in the left sidebar and download")
        print("gate1_results.zip manually.")

---

## Done

Paste back into the chat:

1. The **GATE 1 — DISCOVERY REPORT** block from Step 5, including the NBBTRADER channel resolution section at the bottom of it
2. Anything from Step 6 that looks like it was wrongly excluded
3. The zip from Step 7 if it downloaded

Then we move on to pulling the transcripts.